## Population Characteristics Analysis

* Descriptive statistics of the CGMacros study population from bio.csv

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('..').resolve()

DATA_ROOT = PROJECT_ROOT / 'CGMacros'
OUT_DIR = PROJECT_ROOT / 'dataset' / 'analysis_outputs' / 'bio'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"IN_COLAB={IN_COLAB}  DATA_ROOT={DATA_ROOT}  OUT_DIR={OUT_DIR}")

Mounted at /content/drive
IN_COLAB=True  DATA_ROOT=/content/drive/MyDrive/dissertation/CGMacros  OUT_DIR=/content/drive/MyDrive/dissertation/dataset/analysis_outputs/bio


In [2]:
import json

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

summary = {}

### 1. Dataset overview

In [3]:
bio = pd.read_csv(DATA_ROOT / 'bio.csv')
bio['participant_id'] = bio['subject'].apply(lambda s: f'CGMacros-{s:03d}')

def glycemic_group(a1c):
    if a1c < 5.7:
        return 'Healthy'
    elif a1c < 6.5:
        return 'Prediabetes'
    return 'Type2Diabetes'

bio['group'] = bio['A1c PDL (Lab)'].apply(glycemic_group)
bio = bio.set_index('participant_id')

print(f'number of participants: {len(bio)}')
print(f'\nglycaemic group counts: ')
print(bio['group'].value_counts())

print(f'\ncorresponding features: ')
print(f'{bio.shape[1]} columns:')
print(bio.dtypes)

summary['n_cohort'] = len(bio)
summary['glycemic_group_counts'] = bio['group'].value_counts().to_dict()

number of participants: 45

glycaemic group counts: 
group
Prediabetes      16
Healthy          15
Type2Diabetes    14
Name: count, dtype: int64

corresponding features: 
25 columns:
subject                          int64
Age                              int64
Gender                          object
BMI                            float64
Body weight                    float64
Height                         float64
Self-identify                   object
A1c PDL (Lab)                  float64
Fasting GLU - PDL (Lab)          int64
Insulin                        float64
Triglycerides                    int64
Cholesterol                      int64
HDL                              int64
Non HDL                          int64
LDL (Cal)                        int64
VLDL (Cal)                       int64
Cho/HDL Ratio                  float64
Collection time PDL (Lab)       object
#1 Contour Fingerstick GLU       int64
Time (t)                        object
 #2 Contour Fingerstick GLU      int6

### 2. Demographics


In [4]:
GROUPS = ['Healthy', 'Prediabetes', 'Type2Diabetes']
demo_cont_vars = ['Age', 'BMI', 'Body weight ', 'Height ']

print('Shapiro-Wilk normality check (p < 0.05 => reject normality):')
for var in demo_cont_vars:
    for grp in GROUPS:
        vals = bio.loc[bio['group'] == grp, var].dropna()
        stat, p = stats.shapiro(vals)
        flag = '  <-- non-normal' if p < 0.05 else ''
        print(f'  {var:15s} {grp:15s} n={len(vals):2d}  p={p:.3f}{flag}')

Shapiro-Wilk normality check (p < 0.05 => reject normality):
  Age             Healthy         n=15  p=0.445
  Age             Prediabetes     n=16  p=0.328
  Age             Type2Diabetes   n=14  p=0.145
  BMI             Healthy         n=15  p=0.051
  BMI             Prediabetes     n=16  p=0.840
  BMI             Type2Diabetes   n=14  p=0.239
  Body weight     Healthy         n=15  p=0.133
  Body weight     Prediabetes     n=16  p=0.583
  Body weight     Type2Diabetes   n=14  p=0.746
  Height          Healthy         n=15  p=0.778
  Height          Prediabetes     n=16  p=0.152
  Height          Type2Diabetes   n=14  p=0.460


In [5]:
def describe_continuous(var):
    group_vals = [bio.loc[bio['group'] == g, var].dropna() for g in GROUPS]
    row = {'variable': var.strip()}
    for grp, vals in zip(GROUPS, group_vals):
        row[f'{grp} mean (SD)'] = f'{vals.mean():.1f} ({vals.std():.1f})'
        row[f'{grp} median [IQR]'] = f'{vals.median():.1f} [{vals.quantile(.25):.1f}-{vals.quantile(.75):.1f}]'
    row['Kruskal-Wallis p'] = stats.kruskal(*group_vals).pvalue
    row['ANOVA p'] = stats.f_oneway(*group_vals).pvalue
    return row

table1_cont = pd.DataFrame([describe_continuous(v) for v in demo_cont_vars])
pd.set_option('display.width', 160)
print(table1_cont.to_string(index=False))

summary['table1_continuous'] = table1_cont.to_dict(orient='records')

   variable Healthy mean (SD) Healthy median [IQR] Prediabetes mean (SD) Prediabetes median [IQR] Type2Diabetes mean (SD) Type2Diabetes median [IQR]  Kruskal-Wallis p  ANOVA p
        Age       38.7 (14.5)     40.0 [25.5-48.0]           51.6 (10.7)         54.0 [45.8-58.2]              54.2 (5.6)           52.5 [51.0-59.0]          0.005676 0.000845
        BMI        28.1 (6.4)     25.8 [23.0-30.5]            33.4 (7.5)         32.8 [28.8-36.2]              31.8 (5.3)           30.6 [28.2-34.3]          0.069142 0.082080
Body weight      173.4 (40.1)  169.2 [141.6-191.0]          191.4 (41.1)      191.0 [163.7-206.1]            182.0 (22.5)        180.9 [168.6-197.7]          0.259666 0.386906
     Height        65.8 (3.2)     66.0 [63.2-67.5]            63.6 (3.4)         62.8 [61.8-66.2]              63.7 (2.8)           64.0 [61.2-65.5]          0.105881 0.100882


In [6]:
demo_cat_vars = ['Gender', 'Self-identify ']

for var in demo_cat_vars:
    ct = pd.crosstab(bio[var], bio['group'])[GROUPS]
    print(f'--- {var.strip()} ---')
    print(ct)
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    min_expected = expected.min()
    print(f'chi2 p={p:.3f}  (min expected cell count={min_expected:.2f}'
          f'{" -- below 5, chi2 assumption violated, treat as descriptive only" if min_expected < 5 else ""})')
    print()
    summary.setdefault('table1_categorical', {})[var.strip()] = {
        'counts': ct.to_dict(), 'chi2_p': float(p), 'min_expected_count': float(min_expected),
    }

--- Gender ---
group   Healthy  Prediabetes  Type2Diabetes
Gender                                     
F            10           11              8
M             5            5              6
chi2 p=0.784  (min expected cell count=4.98 -- below 5, chi2 assumption violated, treat as descriptive only)

--- Self-identify ---
group                    Healthy  Prediabetes  Type2Diabetes
Self-identify                                               
African American               1            2              0
Black, African American        0            1              0
Hispanic/Latino                8           12             14
White                          6            1              0
chi2 p=0.025  (min expected cell count=0.31 -- below 5, chi2 assumption violated, treat as descriptive only)



### 3. Clinical Indicators

In [7]:
lab_vars = ['A1c PDL (Lab)', 'Fasting GLU - PDL (Lab)', 'Insulin ', 'Triglycerides',
            'Cholesterol', 'HDL', 'Non HDL ', 'LDL (Cal)', 'VLDL (Cal)', 'Cho/HDL Ratio']

print('Shapiro-Wilk normality check (p < 0.05 => reject normality):')
for var in lab_vars:
    for grp in GROUPS:
        vals = bio.loc[bio['group'] == grp, var].dropna()
        stat, p = stats.shapiro(vals)
        flag = '  <-- non-normal' if p < 0.05 else ''
        print(f'  {var:26s} {grp:15s} n={len(vals):2d}  p={p:.3f}{flag}')

Shapiro-Wilk normality check (p < 0.05 => reject normality):
  A1c PDL (Lab)              Healthy         n=15  p=0.104
  A1c PDL (Lab)              Prediabetes     n=16  p=0.383
  A1c PDL (Lab)              Type2Diabetes   n=14  p=0.046  <-- non-normal
  Fasting GLU - PDL (Lab)    Healthy         n=15  p=0.850
  Fasting GLU - PDL (Lab)    Prediabetes     n=16  p=0.124
  Fasting GLU - PDL (Lab)    Type2Diabetes   n=14  p=0.380
  Insulin                    Healthy         n=15  p=0.192
  Insulin                    Prediabetes     n=16  p=0.037  <-- non-normal
  Insulin                    Type2Diabetes   n=14  p=0.392
  Triglycerides              Healthy         n=15  p=0.001  <-- non-normal
  Triglycerides              Prediabetes     n=16  p=0.014  <-- non-normal
  Triglycerides              Type2Diabetes   n=14  p=0.000  <-- non-normal
  Cholesterol                Healthy         n=15  p=0.640
  Cholesterol                Prediabetes     n=16  p=0.081
  Cholesterol                Type

In [8]:
table1_labs = pd.DataFrame([describe_continuous(v) for v in lab_vars])
print(table1_labs.to_string(index=False))

summary['table1_labs'] = table1_labs.to_dict(orient='records')

               variable Healthy mean (SD) Healthy median [IQR] Prediabetes mean (SD) Prediabetes median [IQR] Type2Diabetes mean (SD) Type2Diabetes median [IQR]  Kruskal-Wallis p      ANOVA p
          A1c PDL (Lab)         5.2 (0.3)        5.3 [5.0-5.5]             6.0 (0.2)            6.0 [5.8-6.1]               7.2 (0.6)              7.1 [6.9-7.4]      3.150459e-09 1.267397e-16
Fasting GLU - PDL (Lab)        95.7 (9.5)    96.0 [90.5-101.0]          116.2 (15.6)      110.5 [104.0-125.2]            152.6 (29.6)        149.0 [142.5-157.0]      4.113095e-07 5.630225e-09
                Insulin         9.7 (5.7)       9.3 [5.2-13.2]            17.9 (9.9)         17.5 [13.0-20.3]              14.8 (6.6)           13.2 [10.6-17.5]      1.373355e-02 1.690330e-02
          Triglycerides      102.9 (62.9)    83.0 [64.0-122.0]          139.1 (69.6)       120.0 [98.0-148.5]           245.1 (274.3)        160.5 [124.5-248.0]      7.840844e-03 6.142187e-02
            Cholesterol      186.5 (45.7

In [9]:
outlier_cols = ['Triglycerides', 'Cholesterol', 'HDL', 'Non HDL ', 'LDL (Cal)', 'VLDL (Cal)',
                'Cho/HDL Ratio']
t2d = bio[bio['group'] == 'Type2Diabetes']
print(t2d[outlier_cols].to_string())

implied_vldl = t2d['Triglycerides'] / 5
implied_ldl = t2d['Cholesterol'] - t2d['HDL'] - t2d['VLDL (Cal)']
bad = t2d[(implied_vldl - t2d['VLDL (Cal)']).abs() > 20]
print('\nrows where VLDL (Cal) is inconsistent with Triglycerides / 5:')
print(bad[['subject', 'Triglycerides', 'VLDL (Cal)']].assign(implied_vldl=implied_vldl[bad.index]))

                Triglycerides  Cholesterol  HDL  Non HDL   LDL (Cal)  VLDL (Cal)  Cho/HDL Ratio
participant_id                                                                                 
CGMacros-003              154          190   74       116         90          31            2.6
CGMacros-005              392          269   38       231        157          78            7.1
CGMacros-012             1150          213   24       189        800         400          400.0
CGMacros-014               88          142   44        98         81          18            3.2
CGMacros-028              260          302   56       246        195          52            5.4
CGMacros-030              249          226   38       188        143          50            5.9
CGMacros-035               92          188   47       141        124          18            4.0
CGMacros-036              128          116   60        56         34          26            1.9
CGMacros-038              245          1

### 4. Missing Value Check

In [10]:
na_counts = bio.isna().sum()
print('missing values per column (whole cohort):')
print(na_counts[na_counts > 0] if na_counts.any() else '(none -- bio.csv has zero missing values)')

na_by_group = bio.drop(columns='group').isna().groupby(bio['group']).sum()
print('\nmissing values per column, by group (all zero, shown for completeness):')
print(na_by_group.loc[:, na_by_group.sum() > 0] if na_by_group.values.any() else '(none in any group)')

summary['n_missing_total'] = int(na_counts.sum())

missing values per column (whole cohort):
(none -- bio.csv has zero missing values)

missing values per column, by group (all zero, shown for completeness):
(none in any group)


### Save Summary

In [11]:
out_path = OUT_DIR / 'bio_analysis_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'saved to {out_path}')

saved to /content/drive/MyDrive/dissertation/dataset/analysis_outputs/bio/bio_analysis_summary.json
